## Question: Can simple price, momentum, volatility, and trading-activity features predict relative next-day equity returns out of sample, and does nonlinear machine learning provide economically meaningful improvements over regularized linear models after accounting for turnover and transaction costs?

---

# Data Pipeline

### Objective: Build a reproducible, leakage-free panel dataset for cross-sectional equity return prediction that

- downloads daily OHLCV data for a fixed universe of liquid U.S. equities
- retains an unbalanced panel rather than requiring every ticker on every date, i.e., we do not drop data for all the tickers on a particular date just because one of the ticker data is missing on that date. The pipeline does not force an inner join across tickers; the frozen dataset used in this project was verified to have complete common trading-date coverage for all 31 symbols.
- calculates daily returns using adjusted close prices
- constructs next-day forward returns
- constructs a cross-sectionally demeaned next-day return target
- assigns chronological train / validation / test periods
- ensures that information only at time $t$ is used to predict returns after time $t$

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

# Making this repo portable rather than using an absolute windows path to import files
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

sys.path.append(str(SRC_DIR))

from config import (
    UNIVERSE,
    MARKET_TICKER,
    START_DATE,
    END_DATE,
    TRAIN_END,
    VAL_END,
)

DATA_DIR.mkdir(exist_ok=True)

#### For computing returns/rolling features, I will sort the data by (ticker , date) inside the functions, becasue say shift() must be applied in chronological order within each ticker.

#### The final data returned will be sorted by ( date, ticker) as it helps in time slicing and having time flow top-to-bottom is easier to inspect and debug. It also matches with how panel data is usually viewed: "For each date, here are all assets."

#### The output of every function will be sorted by (date , ticker) and it will be converted into (ticker, date) internally inside a function that uses shift, rolling, or anything order-dependent per ticker before processing it. This way I get correct computation and a dataset that is easy to inspect and slice by time.

In [2]:
""" Functions defined for sorting"""


def sort_date_ticker(df):
    return df.sort_values(["date", "ticker"]).reset_index(drop=True)


def sort_ticker_date(df):
    return df.sort_values(["ticker", "date"]).reset_index(drop=True)

In [3]:
def download_ohlcv(tickers, start, end=None):
    """
    Download daily OHLCV data from Yahoo Finance and return
    a tidy panel with one row per (date, ticker).

    Returns columns:  date  ticker  open  high  low  close  adj_close  volume
    """

    df = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,
        actions=False,
        group_by="column",
        progress=False,
        threads=True,
    )

    tidy = []

    if isinstance(df.columns, pd.MultiIndex):
        available_tickers = set(df.columns.get_level_values(1))

        for ticker in tickers:
            if ticker not in available_tickers:
                print(f"Warning: {ticker} was not returned by yfinance.")
                continue

            sub = df.xs(ticker, axis=1, level=1).copy()

            sub.columns = [c.lower().replace(" ", "_") for c in sub.columns]

            sub["ticker"] = ticker
            sub = sub.reset_index()

            tidy.append(sub)

        out = pd.concat(tidy, ignore_index=True)

    else:
        out = df.copy()

        out.columns = [c.lower().replace(" ", "_") for c in out.columns]

        out = out.reset_index()
        out["ticker"] = tickers[0]

    out = out.rename(
        columns={
            "Date": "date",
            "adjclose": "adj_close",
        }
    )

    needed = {
        "date",
        "ticker",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume",
    }

    missing = needed - set(out.columns)

    if missing:
        raise ValueError(f"Missing columns from yfinance output: {missing}")

    return sort_date_ticker(out)

In [4]:
ALL_TICKERS = UNIVERSE + [MARKET_TICKER]

raw = download_ohlcv(tickers=ALL_TICKERS, start=START_DATE, end = END_DATE)

print(f"Raw shape of the data = {raw.shape} \nNumber of unique tickers in the data (should be 31) = {raw['ticker'].nunique()}")
raw.head()

Raw shape of the data = (129828, 8) 
Number of unique tickers in the data (should be 31) = 31


,date,adj_close,close,high,low,open,volume,ticker
0,2010-01-04,6.400960,7.643214,7.660714,7.585000,7.622500,493729600,AAPL
1,2010-01-04,38.523415,57.720001,57.869999,56.560001,56.630001,5277400,AMGN
2,2010-01-04,43.777538,56.180000,56.389999,54.799999,55.720001,6186700,BA
3,2010-01-04,11.977122,15.690000,15.750000,15.120000,15.240000,180845200,BAC
4,2010-01-04,24.852760,34.000000,34.099998,33.400002,33.599998,40679350,C


In [5]:
# SAve the raw data to avoid downloading the data from yfinance repeatedly
raw.to_csv(DATA_DIR / "raw_prices.csv", index=False)

In [6]:
# load the saved raw data
raw = pd.read_csv(DATA_DIR / "raw_prices.csv", parse_dates=["date"])

In [7]:
def clean_data(raw):
    """
    Clean the OHLCV panel without forcing all tickers to share an identical calendar.

    - standardize date type
    - remove duplicate (date, ticker) rows
    - remove rows without usable adjusted prices
    - sort by date/ticker
    """

    out = raw.copy()

    out["date"] = pd.to_datetime(out["date"])

    out = out.drop_duplicates(subset=["date", "ticker"])

    out = out.dropna(subset=["adj_close"])

    out = out[out["adj_close"] > 0]

    return sort_date_ticker(out)

In [8]:
prices = clean_data(raw)

print(f"The number of duplicate (date, ticker) rows = {prices.duplicated(['date', 'ticker']).sum()}")

The number of duplicate (date, ticker) rows = 0


In [9]:
coverage = (
    prices.groupby("ticker")
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        n_obs=("date", "size"),
    )
    .sort_values("n_obs")
)

coverage

,start_date,end_date,n_obs
ticker,,,
AAPL,2010-01-04,2026-08-27,4188
AMGN,2010-01-04,2026-08-27,4188
BA,2010-01-04,2026-08-27,4188
BAC,2010-01-04,2026-08-27,4188
C,2010-01-04,2026-08-27,4188
CAT,2010-01-04,2026-08-27,4188
CSCO,2010-01-04,2026-08-27,4188
CVX,2010-01-04,2026-08-27,4188
GS,2010-01-04,2026-08-27,4188


In [10]:
prices.groupby("date")["ticker"].nunique().describe()

count    4188.0
mean       31.0
std         0.0
min        31.0
25%        31.0
50%        31.0
75%        31.0
max        31.0
Name: ticker, dtype: float64

In [11]:
def add_returns(df):
    """
    Add simple daily returns using adjusted close prices.

    ret_1d(t) = P_t / P_{t-1} - 1
    """

    out = sort_ticker_date(df.copy())

    out["ret_1d"] = out.groupby("ticker")["adj_close"].pct_change(fill_method=None)

    return sort_date_ticker(out)

In [12]:
prices = add_returns(prices)

prices[prices["ticker"] == "AAPL"][["date", "adj_close", "ret_1d"]].head()

,date,adj_close,ret_1d
0,2010-01-04,6.400960,NaN
31,2010-01-05,6.412027,0.001729
62,2010-01-06,6.310034,-0.015907
93,2010-01-07,6.298368,-0.001849
124,2010-01-08,6.340244,0.006649


## To find the forward target return we can't just shift by one like in the volatility modeling project as some of the tickers could be missing values for a particular date.
Example: If AAPL is missing on Tuesday, a shift(-1) would tell us that the Wednesday's return is the "next-day" return after Monday.
To deal with this, we build a market trading calendar using the SPY data

In [13]:
def make_market_calendar(df, market_ticker="SPY"):
    """
    Build a mapping:

    date -> next_market_date

    using the market reference ticker.
    """

    market_dates = df.loc[df["ticker"] == market_ticker, ["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

    market_dates["next_market_date"] = market_dates["date"].shift(-1)

    return market_dates

In [14]:
calendar = make_market_calendar(prices, MARKET_TICKER)

calendar.head(10)

,date,next_market_date
0,2010-01-04,2010-01-05
1,2010-01-05,2010-01-06
2,2010-01-06,2010-01-07
3,2010-01-07,2010-01-08
4,2010-01-08,2010-01-11
5,2010-01-11,2010-01-12
6,2010-01-12,2010-01-13
7,2010-01-13,2010-01-14
8,2010-01-14,2010-01-15
9,2010-01-15,2010-01-19


In [15]:
# Now we construct the targets by exluding the SPY from the actual modeling universe
stocks = prices[prices["ticker"].isin(UNIVERSE)].copy()


def add_target(df, market_calendar):
    """
    Create leakage-safe next-trading-day return targets.

    fwd_ret_1d:
        actual stock return on the next market trading day.

    target_cs_1d:
        next-day stock return minus the cross-sectional
        average next-day return of the universe.
    """

    out = sort_ticker_date(df.copy())

    # Date of the stock's next available observation
    out["next_obs_date"] = out.groupby("ticker")["date"].shift(-1)

    # Return observed at that next observation
    out["fwd_ret_1d"] = out.groupby("ticker")["ret_1d"].shift(-1)

    # Attach true next market trading date
    out = out.merge(market_calendar, on="date", how="left")

    # If the stock's next observation is not actually
    # the next market day, do not call it a 1-day target.
    valid_next_day = out["next_obs_date"] == out["next_market_date"]

    out.loc[~valid_next_day, "fwd_ret_1d"] = np.nan

    # Cross-sectional average future return on each date
    cs_mean = out.groupby("date")["fwd_ret_1d"].transform("mean")

    out["target_cs_1d"] = out["fwd_ret_1d"] - cs_mean

    return sort_date_ticker(out)

In [16]:
stocks = add_target(stocks, calendar)

In [17]:
# Add Train, Validation and Test labels
def add_split_labels(df, train_end, val_end):
    """
    Assign chronological dataset splits.

    Train:
        date <= train_end

    Validation:
        train_end < date <= val_end

    Test:
        date > val_end
    """

    out = df.copy()

    train_end = pd.Timestamp(train_end)
    val_end = pd.Timestamp(val_end)

    out["split"] = np.select(
        [
            out["date"] <= train_end,
            ((out["date"] > train_end) & (out["date"] <= val_end)),
            out["date"] > val_end,
        ],
        [
            "train",
            "validation",
            "test",
        ],
        default="unknown",
    )

    return out

In [18]:
stocks = add_split_labels(stocks, TRAIN_END, VAL_END)

split_table = stocks.groupby("split")["date"].agg(["min", "max", "nunique"]).reindex(["train", "validation", "test"])

split_table

,min,max,nunique
split,,,
train,2010-01-04,2018-12-31,2264
validation,2019-01-02,2021-12-31,757
test,2022-01-03,2026-08-27,1167


In [19]:
# Drop unusable target rows
base = stocks.dropna(subset=["fwd_ret_1d", "target_cs_1d"]).copy()


In [20]:
# Sanity Checks

# Unique Key
assert not base.duplicated(["date", "ticker"]).any()

# Universe Only
assert set(base["ticker"].unique()).issubset(set(UNIVERSE))

# No SPY accidentally included
assert MARKET_TICKER not in base["ticker"].unique()

# Cross-sectional target should average approximately zero each day
(base.groupby("date")["target_cs_1d"].mean().abs().describe())


count    4.187000e+03
mean     3.851311e-19
std      5.425020e-19
min      0.000000e+00
25%      1.156482e-19
50%      2.312965e-19
75%      4.625929e-19
max      1.387779e-17
Name: target_cs_1d, dtype: float64

In [21]:
# Number of stocks each day
(base.groupby("date")["ticker"].nunique().describe())

count    4187.0
mean       30.0
std         0.0
min        30.0
25%        30.0
50%        30.0
75%        30.0
max        30.0
Name: ticker, dtype: float64

In [22]:
base[base["ticker"] == "AAPL"][["date", "ret_1d", "fwd_ret_1d", "target_cs_1d"]].head(10)

,date,ret_1d,fwd_ret_1d,target_cs_1d
0,2010-01-04,NaN,0.001729,-0.004657
30,2010-01-05,0.001729,-0.015907,-0.016047
60,2010-01-06,-0.015907,-0.001849,-0.008736
90,2010-01-07,-0.001849,0.006649,0.007218
120,2010-01-08,0.006649,-0.008822,-0.010461
150,2010-01-11,-0.008822,-0.011375,-0.000655
180,2010-01-12,-0.011375,0.014105,0.005299
210,2010-01-13,0.014105,-0.005791,-0.011153
240,2010-01-14,-0.005791,-0.016712,-0.005213
270,2010-01-15,-0.016712,0.044238,0.030532


In [23]:
base.columns

Index(['date', 'adj_close', 'close', 'high', 'low', 'open', 'volume', 'ticker',
       'ret_1d', 'next_obs_date', 'fwd_ret_1d', 'next_market_date',
       'target_cs_1d', 'split'],
      dtype='object')

In [24]:
base.to_csv(DATA_DIR / "alpha_dataset_base.csv", index=False)

In [25]:
base.head(10)

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,fwd_ret_1d,next_market_date,target_cs_1d,split
0,2010-01-04,6.400960,7.643214,7.660714,7.585000,7.622500,493729600,AAPL,NaN,2010-01-05,0.001729,2010-01-05,-0.004657,train
1,2010-01-04,38.523415,57.720001,57.869999,56.560001,56.630001,5277400,AMGN,NaN,2010-01-05,-0.008663,2010-01-05,-0.015049,train
2,2010-01-04,43.777538,56.180000,56.389999,54.799999,55.720001,6186700,BA,NaN,2010-01-05,0.032752,2010-01-05,0.026366,train
3,2010-01-04,11.977122,15.690000,15.750000,15.120000,15.240000,180845200,BAC,NaN,2010-01-05,0.032505,2010-01-05,0.026119,train
4,2010-01-04,24.852760,34.000000,34.099998,33.400002,33.599998,40679350,C,NaN,2010-01-05,0.038235,2010-01-05,0.031849,train
5,2010-01-04,39.051941,58.549999,59.189999,57.509998,57.650002,7325600,CAT,NaN,2010-01-05,0.011955,2010-01-05,0.005569,train
6,2010-01-04,15.830381,24.690001,24.840000,24.010000,24.110001,59853700,CSCO,NaN,2010-01-05,-0.004455,2010-01-05,-0.010842,train
7,2010-01-04,40.537148,79.059998,79.199997,78.160004,78.199997,10173800,CVX,NaN,2010-01-05,0.007083,2010-01-05,0.000697,train
8,2010-01-04,128.756821,173.080002,174.250000,169.509995,170.050003,9135000,GS,NaN,2010-01-05,0.017679,2010-01-05,0.011293,train
9,2010-01-04,19.155914,28.670000,29.150000,28.549999,29.150000,13120900,HD,NaN,2010-01-05,0.007324,2010-01-05,0.000938,train
